# Expression from H&E with a frozen pathology encoder

This guide predicts gene expression from H&E images for one tissue, with a recipe adapted from the HEST-Benchmark: patches of 112 µm around each Visium spot, embeddings from the frozen encoder H-optimus-0, then PCA and ridge regression. It evaluates on held-out study groups. Most full-resolution H&E sections in spatialdata-db come from HEST-1k, which the client reaches through the same API as all other sources. The guide makes no claim about prediction accuracy.

The catalog release and the tissue are parameters. H-optimus-0 requires accepting its terms on Hugging Face and a token in `HF_TOKEN`.

In [1]:
import os

os.environ.setdefault("TQDM_DISABLE", "1")  # keep progress bars out of the rendered guide
RELEASE = os.environ.get("SDDB_RELEASE", "2026-10-28")
TISSUE = os.environ.get("SDDB_TISSUE", "prostate gland")
ENCODER = "hf-hub:bioptimus/H-optimus-0"
PATCH_UM, PATCH_PX, SPOT_UM, MIN_SIDE, MIN_PANEL, N_GENES, N_PCS, SEED = 112.0, 224, 55.0, 10_000, 5_000, 50, 256, 0
EXCLUDE = {"ADgyHXEKFq7YjJTb0001"}  # conversion defect: downscaled full_image without scale transform

In [2]:
import re

import lightning
import numpy as np
import pandas as pd
import scanpy as sc
import sddb
import timm
import torch
from PIL import Image
from scipy.stats import pearsonr
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
from spatialdata.transformations import get_transformation

lightning.seed_everything(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device", device)

device cuda


## Query and cohort

Human Visium sections of one tissue with a full-resolution H&E image of at least 10,000 pixels. A group is the catalog collection of a dataset, or its study when it has none.

In [3]:
def full_side(r):
    s = [max(int(a), int(b)) for a, b in re.findall(r"'[^']*full_image': DataTree\[[a-z]+\] \(3, (\d+), (\d+)\)", r or "")]
    return max(s) if s else 0


catalog = sddb.Catalog(version=RELEASE)
hits = catalog.query(organism="human", tissue=TISSUE, technology="Visium")
m = hits.members.assign(side=hits.members["sdata_repr"].map(full_side))
m = m[(m["side"] >= MIN_SIDE) & ~m["uid"].isin(EXCLUDE)].copy()
m["group"] = m["collection_name"].fillna(m["study_id"]).fillna(m["uid"]).astype(str)
cohort = catalog.cohort(uids=list(m["uid"]))
cohort.freeze().write("he_cohort.json")
m["canonical_key"].str.split(":").str[0].value_counts(), m["group"].nunique()

(canonical_key
 hest    32
 10x      3
 Name: count, dtype: int64,
 6)

## Patches around spots

For each dataset the client copies only the full-resolution image, the spot shapes and the count table. The pixel size is computed from the Visium spot diameter (55 µm), because the catalog value describes the downscaled image for some datasets.

In [4]:
def xy_affine(el, cs):
    return get_transformation(el, to_coordinate_system=cs).to_affine_matrix(input_axes=("x", "y"), output_axes=("x", "y"))


def patches(ds):
    image = next(p for p in ds.elements() if re.search(r"^images/.*full_image$", p))
    table = f"tables/{ds.default_table or 'table'}"
    t = ds.to_sdata(elements=[table]).tables[table.split("/", 1)[1]]
    attrs = t.uns["spatialdata_attrs"]
    region = attrs["region"] if isinstance(attrs["region"], str) else attrs["region"][0]
    sd = ds.to_sdata(elements=[image, table, f"shapes/{region}"])
    shapes, img = sd.shapes[region], sd.images[image.split("/", 1)[1]]
    img0 = img["scale0"]["image"] if hasattr(img, "children") and "scale0" in img else img
    cs = next(iter(set(get_transformation(shapes, get_all=True)) & set(get_transformation(img0, get_all=True))))
    to_px = np.linalg.inv(xy_affine(img0, cs)) @ xy_affine(shapes, cs)
    if "in_tissue" in t.obs:
        t = t[t.obs["in_tissue"].astype(int) == 1].copy()
    spots = shapes.loc[t.obs[attrs["instance_key"]].values]
    xy = (to_px @ np.c_[spots.geometry.centroid.x, spots.geometry.centroid.y, np.ones(len(spots))].T)[:2].T
    mpp = SPOT_UM / (2 * float(np.median(spots["radius"])) * float(np.sqrt(abs(np.linalg.det(to_px[:2, :2])))))
    arr, half = np.asarray(img0.transpose("y", "x", "c").data.compute()), int(round(PATCH_UM / mpp / 2))
    keep, out = [], []
    for i, (x, y) in enumerate(np.round(xy).astype(int)):
        if half <= x < arr.shape[1] - half and half <= y < arr.shape[0] - half:
            out.append(np.asarray(Image.fromarray(arr[y - half:y + half, x - half:x + half, :3]).resize((PATCH_PX, PATCH_PX), Image.BILINEAR)))  # same filter as he_expression.py
            keep.append(i)
    t = t[keep].copy()
    col = next((c for c in t.var.columns if t.var[c].astype(str).str.startswith("ENSG").mean() > 0.5), None)
    ids = t.var[col].astype(str) if col else pd.Series(t.var_names, index=t.var_names).astype(str)
    t.var_names = ids.str.split(".").str[0].values  # one gene id type (Ensembl) across sources
    return np.stack(out), t[:, ~t.var_names.duplicated()].copy()

## Frozen encoder embeddings

In [5]:
model = timm.create_model(ENCODER, pretrained=True, init_values=1e-5, dynamic_img_size=False).eval().to(device)
cfg = timm.data.resolve_data_config({}, model=model)
mean, std = torch.tensor(cfg["mean"]).view(1, 3, 1, 1), torch.tensor(cfg["std"]).view(1, 3, 1, 1)


def embed(p):
    feats = []
    with torch.inference_mode(), torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
        for b in range(0, len(p), 256):
            x = torch.from_numpy(p[b:b + 256]).permute(0, 3, 1, 2).float().div(255)
            feats.append(model(((x - mean) / std).to(device)).float().cpu().numpy())
    return np.concatenate(feats)


embs, tables = {}, {}
for ds in cohort:
    p, t = patches(ds)
    embs[ds.uid], tables[ds.uid] = embed(p), t
len(embs)

35

## Ridge regression, one held-out group at a time

Genes are those measured in every dataset; datasets with targeted panels of fewer than 5,000 genes are excluded. The 50 most variable genes are chosen on the training groups only.

In [6]:
uids = [u for u in embs if tables[u].n_vars >= MIN_PANEL]
genes = sorted(set.intersection(*[set(tables[u].var_names) for u in uids]))
expr = {}
for u in uids:
    a = tables[u][:, genes].copy()
    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)
    expr[u] = np.asarray(a.X.todense() if hasattr(a.X, "todense") else a.X, dtype=np.float32)
group = dict(zip(m["uid"], m["group"]))
rows = []
for g in sorted({group[u] for u in uids}):
    te, tr = [u for u in uids if group[u] == g], [u for u in uids if group[u] != g]
    ytr, xtr = np.concatenate([expr[u] for u in tr]), np.concatenate([embs[u] for u in tr])
    top = np.argsort(ytr.var(0))[::-1][:N_GENES]
    pca = PCA(n_components=min(N_PCS, len(xtr) - 1), random_state=SEED).fit(xtr)
    ridge = RidgeCV(alphas=np.logspace(-2, 4, 13)).fit(pca.transform(xtr), ytr[:, top])
    pred = ridge.predict(pca.transform(np.concatenate([embs[u] for u in te])))
    yte = np.concatenate([expr[u] for u in te])[:, top]
    rows += [{"held_out_group": g, "gene": genes[i], "pearson": pearsonr(pred[:, j], yte[:, j])[0]}
             for j, i in enumerate(top) if yte[:, j].std() > 0]
res = pd.DataFrame(rows)
res.groupby("held_out_group")["pearson"].mean().round(3)

held_out_group
10x:human-prostate-cancer-acinar-cell-carcinoma-ffpe-1-standard                           0.248
10x:human-prostate-cancer-adenocarcinoma-with-invasive-carcinoma-ffpe-1-standard-1-3-0    0.406
10x:human-prostate-cancer-adjacent-normal-section-with-if-staining-ffpe-1-standard        0.252
doi-10-1038-s41467-023-36071-5                                                            0.415
doi-10-1038-s41586-022-05023-2                                                            0.340
hest-prostate-st-internal                                                                 0.295
Name: pearson, dtype: float64